# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response   
to the last marketing campaign.   
Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [242]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df["total_claim_amount"] = pd.to_numeric(df["total_claim_amount"], errors="coerce")
df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")
df["monthly_premium_auto"] = pd.to_numeric(df["monthly_premium_auto"], errors="coerce")

# Filtro simultáneo: total_claim_amount < 1000 y response == "Yes"
low_claim_yes = df[                                 # el dataframe buscado es este
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
]

low_claim_yes.head()



,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,location_code,marital_status,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,49078,Suburban,Single,97,10.0,3,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,66839,Suburban,Single,154,33.0,24,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,79487,Suburban,Divorced,114,20.0,87,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,41479,Rural,Married,94,14.0,38,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,63774,Rural,Divorced,111,18.0,63,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [243]:
yes_df = df[df["response"] == "Yes"]
pivot_yes = yes_df.pivot_table(
    values=["monthly_premium_auto",     # monthly_premium no existe , la columna es monthly_premium_auto
                                        # Prima mensual alta → genera ingresos constantes 
            "customer_lifetime_value"]  # CLV alto → aporta mucho valor a largo plazo
    , 
    index="policy_type",                                        
    columns="gender",
    aggfunc="mean"
).round(2)

pivot_yes



customer_lifetime_value          monthly_premium_auto       
gender                               F        M                    F      M
policy_type                                                                
Corporate Auto                 7712.63  7944.47                94.30  92.19
Personal Auto                  8339.79  7448.38                99.00  91.09
Special Auto                   7691.58  8247.09                92.31  86.34

Personal_Auto Mujeres es la mas rentable con diferencia
El segundo CLV mas elevado es tambien de Personal Auto mujeres , por lo que Personal Auto Mujeres es el seguro mas rentable
Con la misma logica el seguro menos rentable es Personal Auto de Hombres 

In [244]:
pivot_claim = yes_df.pivot_table(
    values="total_claim_amount",   # Reclamaciones bajas → cuesta poco a la empresa
    index="policy_type",
    columns="gender",
    aggfunc="mean"
).round(2)

pivot_claim


gender,F,M
policy_type,,
Corporate Auto,433.74,408.58
Personal Auto,452.97,457.01
Special Auto,453.28,429.53


El cliente mas ventajoso para la empresa el Corporate Auto de Mujeres
El cliente con mas desventaja es Personal Auto de Hombres 

---

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [245]:
customers_by_state = df["state"].value_counts() # Series de pandas.
filtered_states = customers_by_state[customers_by_state > 500] # Serie de pandas

customers_by_state_df = filtered_states.reset_index()  # convierte esa serie a Dataframe

customers_by_state_df


,state,count
0,California,3552
1,Oregon,2909
2,Arizona,1937
3,Nevada,993
4,Washington,888


 Hay 3 estados que tienen la mayoria con diferencia de seguros, destacando California sobre todos y Oregon tambien tiene muchas polizas.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.
(funcion = maximum, minimum, and median) ( value = customer lifetime value) ( index = level and gender )

In [246]:
clv_stats = df.pivot_table(
    values="customer_lifetime_value",
    index=["education", "gender"],
    aggfunc=["min", "median", "max"]
).round(2)

clv_stats

min                  median  \
                            customer_lifetime_value customer_lifetime_value   
education            gender                                                   
Bachelor             F                      1904.00                 5640.51   
                     M                      1898.01                 5548.03   
College              F                      1898.68                 5623.61   
                     M                      1918.12                 6005.85   
Doctor               F                      2395.57                 5332.46   
                     M                      2267.60                 5577.67   
High School or Below F                      2144.92                 6039.55   
                     M                      1940.98                 6286.73   
Master               F                      2417.78                 5729.86   
                     M                      2272.31                 5579.10   

                                                max  
                            customer_lifetime_value  
education            gender                          
Bachelor             F                     73225.96  
                     M                     67907.27  
College              F                     61850.19  
                     M                     61134.68  
Doctor               F                     44856.11  
                     M                     32677.34  
High School or Below F                     55277.45  
                     M                     83325.38  
Master               F                     51016.07  
                     M                     50568.26

### 1. Grupo con mayor mediana
el grupo con la mediana más alta es el que tiene clientes que, de media, aportan más valor de forma constante.
Por tanto el Grupo mas valioso es el cliente de High School or Below de Hombres

#### 2. Grupo con mayor máximo
El máximo te dice cuál es el cliente más valioso dentro de cada grupo, el cliente estrella
Aqui repite como mas valioso el cliente de High School or Below de Hombres
#### 3. Grupo con menor mínimo
El mínimo muestra el cliente menos valioso del grupo, que aporta poco al seguro.
Los clientes Bachelor y College de ambos sexos son poco valiosos para los seguros


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [247]:
# month no existe 
    
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")  # convierte a formato fecha  2011-02-18
df["month"] = df["effective_to_date"].dt.month  # Extrae el mes del mes


C:\Users\ernes\AppData\Local\Temp\ipykernel_47684\1074228140.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")  # convierte a formato fecha  2011-02-18


In [248]:

policies_month = df.pivot_table(
    values="customer",     # vale cualquiera para contar
    index="state",         
    columns="month",       
    aggfunc="count"
).fillna(0).astype(int)  # fillna(0) para que si un estadp no tiene de  0  y astype(int) para que no salgan decimales

policies_month    # solo tenemos datos de 2 meses (1 = Enero , 2 = Febrero)



month,1,2
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


El estado con mas polizas ha sido California durante el mes de Enero.  
Por numero de poizas al mes los mas rentables son los estados de California y Oregon

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [249]:
df.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type', 'month'],
      dtype='str')

In [250]:
top3_states = df["state"].value_counts().head(3).index
top3 = top3_states.tolist()
top3

['California', 'Oregon', 'Arizona']

In [251]:
policies_month = df.pivot_table(
    values="customer",     
    index=  "month",       
    columns="state",       
    aggfunc="count"
).fillna(0).astype(int) 

policies_month[['California', 'Oregon', 'Arizona']]
    

state,California,Oregon,Arizona
month,,,
1,1918,1565,1008
2,1634,1344,929


Aqui se confirma que el mejor estado para vender seguros es California , casi el doble que el tercero que mas seguros hizo

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

nota: esta es comparando cada canal de marketing con su tasa de respuesta.

In [252]:
# Total de clientes por canal
total_by_channel = df.groupby("sales_channel")["response"].size()

# Total de "Yes" por canal
yes_by_channel = df[df["response"] == "Yes"].groupby("sales_channel")["response"].count()

# Total de "No" por canal
no_by_channel = df[df["response"] == "No"].groupby("sales_channel")["response"].count()

# Total de NaN por canal
nan_by_channel = df[df["response"].isna()].groupby("sales_channel")["response"].size()  # por que count no cuenta nan



# Tasa de respuesta
response_rate = (yes_by_channel / total_by_channel * 100).round(2)

marketing_effect = pd.DataFrame({
    "total_customers": total_by_channel,
    "yes_responses": yes_by_channel,
    "no_responses": no_by_channel,
    "nan_responses": nan_by_channel,
    "response_rate_%": response_rate
}).fillna(0)

marketing_effect


,total_customers,yes_responses,no_responses,nan_responses,response_rate_%
sales_channel,,,,,
Agent,4121,742,3148,231,18.01
Branch,3022,326,2539,157,10.79
Call Center,2141,221,1792,128,10.32
Web,1626,177,1334,115,10.89


NOTA: Aqui melt no haria falta por que responses tiene 'Yes' 'No' ' 'Nan' 

Agent (contacto directo) 
Branch (sucursal física)
Call Center (teléfono)
Web(online)

El canal que mayor numero de clientes fue Agent. 
Ademas fue el que mayor porcentaje de exito tuvo.

Eso indica que si vas a la gente tienes mas posibilidades de que contraten un seguro.
Por telefono , online o que se acerquen a la sucursal poco exito.